# Evaluation

All retrieval evaluation for the thesis. It runs entirely from the two `experiment.jsonl` files produced by the pipeline, with no GPU.

**Setup:**
- 1. Run the full pipeline.ipynb twice, once for the non-pseudonymized arm and once for the pseudonymized arm. 
- 2. Set the two folder paths in the setup cell below to your preferred pseudonymized and non-pseudonymzed experiment names.
- 3. Select the pipeline `venv` as this notebook's kernel.
- 4. Run this notebook top to bottom, only one time. We calculate all evaluation metrics for both arms. 


**What is evaluated.** Each summary is both a query and a candidate. A retrieval is correct when it returns another summary of the same work (same `wikidata_id`), and a summary never retrieves itself. The representations scored are:
- **Lexical baselines:** BoW and TF-IDF, over the full text and over the event triggers.
- **Dense encoders:** E5-Mistral and Qwen3-0.6B embed all six conditions (`raw_text`, `events_only`, `temporal`, `causal`, `temporal_causal_independent`, `temporal_causal_joint`); StoryEmbed embeds `raw_text` only.

Note that all dense and sparse vectors are L2-normalized, so cosine similarity is the dot product everywhere.

**Sections:**
1. **Matched intersection:** restrict both arms to the same summaries (I').
2. **Cosine similarity:** pairwise similarity matrices, per arm.
3. **Overall Retrieval Performance:** overall retrieval, non-pseudonymized vs pseudonymized.
4. **Incremental contribution:** effect of adding temporal and causal structure.
5. **Tversky diagnostic:** retrieval from event-set overlap alone.

In [ ]:
# Imports
import os, sys, re, json, collections
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display, Markdown


# This notebook lives in notebooks/, so the project root is the parent of the working directory.
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_EXPERIMENTS = ROOT / "data" / "experiments"

# The two pipeline runs to evaluate. BOTH are required.
NONANON_DIR = DATA_EXPERIMENTS / "experiment_test_9385_20260609_1013"        # non-pseudonymized arm
ANON_DIR    = DATA_EXPERIMENTS / "anon_experiment_test_9385_20260609_1013"   # pseudonymized arm

# Stop early unless both arms are set and each has an experiment.jsonl.
arms = {"non-pseudonymized (NONANON_DIR)": NONANON_DIR, "pseudonymized (ANON_DIR)": ANON_DIR}
missing_arms = [label for label, directory in arms.items() if directory is None]
if missing_arms:
    raise ValueError("Evaluation requires BOTH dataset paths. Missing: " + ", ".join(missing_arms))
for label, directory in arms.items():
    if not (Path(directory) / "experiment.jsonl").exists():
        raise FileNotFoundError(f"{label}: {Path(directory) / 'experiment.jsonl'} not found. "
                                f"Run the full pipeline for this arm before evaluating.")

# Print out the confirmation
print("Evaluation setup OK")
print(f"  non-pseudonymized : {NONANON_DIR}")
print(f"  pseudonymized     : {ANON_DIR}")

### 1. Matched intersection of the two arms (non-pseudonymized and pseudonymized)

**Goal:** Restrict both arms to the same set of summaries, so every later table compares the non-pseudonymized and pseudonymized runs on an identical population.

**How:** Read the `(wikidata_id, summary_id)` keys from both `experiment.jsonl` files, keep only the keys present in both arms, then re-enforce the floor of at least 2 summaries per work. Both `experiment.jsonl` files are then rewritten in place to this matched set (called I'). 

**Important:** This overwrites both `experiment.jsonl` files because this intersection is the single and absolute source of truth of the thesis, so keep backups before running this notebook if needed. 

In [ ]:
# NONANON_DIR and ANON_DIR are defined once in the setup cell at the top of this notebook.
experiment_files = {"non-anon": NONANON_DIR / "experiment.jsonl", "anon": ANON_DIR / "experiment.jsonl"}
id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')

# Read the (wikidata_id, summary_id) of every row; both ids sit deterministically within the first 160 characters.
def read_summary_keys(path):
    keys = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            match = id_pattern.search(line[:160])
            if match: keys.add((match.group(1), match.group(2)))
    return keys

# Build I': keys present in BOTH arms, then keep only works that still have >=2 summaries, so a relevance cluster has at least 2 summaries to be eligible for retrieval.
nonanon_keys, anon_keys = read_summary_keys(experiment_files["non-anon"]), read_summary_keys(experiment_files["anon"])
shared_keys = nonanon_keys & anon_keys
summaries_per_work = collections.Counter(wid for wid, _ in shared_keys)
Iprime = {(wid, sid) for (wid, sid) in shared_keys if summaries_per_work[wid] >= 2}
print(f"non-anon              : {len(nonanon_keys):>5} summaries / {len({wid for wid, _ in nonanon_keys})} stories")
print(f"anon                  : {len(anon_keys):>5} summaries / {len({wid for wid, _ in anon_keys})} stories")
print(f"intersection          : {len(shared_keys):>5} summaries   (non-anon -{len(nonanon_keys)-len(shared_keys)}, anon -{len(anon_keys)-len(shared_keys)})")
print(f"matched set I' (>=2)   : {len(Iprime):>5} summaries / {len({wid for wid, _ in Iprime})} stories   (cluster removal -{len(shared_keys)-len(Iprime)})\n")

# Rewrite each file in place, keeping only the I' rows: stream to a .tmp file, then atomic os.replace so a mid-run failure can never leave a half-written experiment.jsonl.
for arm_label, experiment_path in experiment_files.items():
    total_rows = kept_rows = 0
    tmp_path = experiment_path.with_suffix(experiment_path.suffix + ".tmp")
    with open(experiment_path, encoding="utf-8") as infile, open(tmp_path, "w", encoding="utf-8") as outfile:
        for line in infile:
            match = id_pattern.search(line[:160])
            if not match: continue
            total_rows += 1
            if (match.group(1), match.group(2)) in Iprime:
                outfile.write(line if line.endswith("\n") else line + "\n"); kept_rows += 1
    os.replace(tmp_path, experiment_path)
    print(f"{arm_label:9s}: {experiment_path.name}  {total_rows} -> {kept_rows} rows   (dropped {total_rows-kept_rows})")
print("\nBoth files now contain ONLY the matched intersection I'.")

### 2. Cosine similarity

**Goal:** Build the pairwise similarity matrix for every encoder-condition and every lexical baseline, for both arms (non-pseudonymized and pseudonymized). These matrices are the substrate that all later per-arm retrieval metrics are computed from.

**How:** A helper function does this for one arm: stack each representation's vectors into one matrix and take `X @ X.T`. Every dense embedding and sparse baseline vector is already L2-normalized, so this dot product is the cosine similarity. The diagonal is set to negative infinity so a summary never retrieves itself. We run the function for both arms, so each arm's matrices are saved to its own `similarities.npz` (one in the non-pseudonymized folder, one in the pseudonymized folder).

In [ ]:
# Compute every pairwise cosine-similarity matrix for ONE arm and save them to that arm's similarities.npz.
# Vectors are already L2-normalized, so vectors @ vectors.T is the cosine similarity; the diagonal is set
# to -inf so a summary is never retrieved as its own neighbor.
def compute_similarities(experiment_dir):
    experiment_path   = experiment_dir / "experiment.jsonl"
    similarities_path = experiment_dir / "similarities.npz"
    similarities_path.parent.mkdir(parents=True, exist_ok=True)

    # Load this arm's experiment data
    rows = [json.loads(line) for line in experiment_path.read_text(encoding="utf-8").splitlines() if line]
    n_summaries = len(rows)

    # One similarity matrix per "<encoder>__<condition>" (dense) and per "<baseline>__raw_text" (sparse)
    similarity_matrices: dict[str, np.ndarray] = {}

    # Dense encoders (e5_mistral, qwen3_emb_0p6b, story_emb): stack the vectors and take vectors @ vectors.T
    encoders = sorted({encoder for row in rows for encoder in row.get("embeddings", {})})
    for encoder in encoders:
        conditions = sorted(rows[0]["embeddings"][encoder]["vectors"].keys())
        for condition in conditions:
            vectors = np.asarray(
                [row["embeddings"][encoder]["vectors"][condition] for row in rows],
                dtype=np.float32,
            )
            similarity = vectors @ vectors.T
            np.fill_diagonal(similarity, -np.inf)
            similarity_matrices[f"{encoder}__{condition}"] = similarity

    # Sparse lexical baselines (bow, tfidf, and their event-trigger variants): rebuild the CSR matrix first
    baselines = sorted({baseline for row in rows for baseline in row.get("baselines", {})})
    for baseline in baselines:
        vocab_size = rows[0]["baselines"][baseline]["dim"]
        values, indices, index_pointer = [], [], [0]
        for row in rows:
            vector = row["baselines"][baseline]["vector"]
            values.extend(vector["values"])
            indices.extend(vector["indices"])
            index_pointer.append(len(values))
        matrix = sparse.csr_matrix((values, indices, index_pointer), shape=(n_summaries, vocab_size), dtype=np.float32)
        similarity = (matrix @ matrix.T).toarray()
        np.fill_diagonal(similarity, -np.inf)
        similarity_matrices[f"{baseline}__raw_text"] = similarity

    # Save all matrices for this arm
    np.savez_compressed(similarities_path, **similarity_matrices)

    # Quick diagnostics: off-diagonal mean per matrix (the diagonal is -inf, so mask it out)
    print(f"[{experiment_dir.name}] wrote {len(similarity_matrices)} similarity matrices to {similarities_path.name}  (N={n_summaries})")
    for name, similarity in similarity_matrices.items():
        off_diagonal = similarity[np.isfinite(similarity)]
        print(f"  {name:48s}  shape={similarity.shape}  mean_off_diag={off_diagonal.mean():.4f}  max={off_diagonal.max():.4f}")

# Run for BOTH arms so each one gets its own similarities.npz (independent inputs to the metrics step).
for arm_dir in (NONANON_DIR, ANON_DIR):
    compute_similarities(arm_dir)

### 3. Overall Retrieval Performance for Matched Data (non-pseudonymized vs pseudonymized)

**Goal:** Produce the thesis Table 2: retrieval performance for every baseline and structural condition, with the non-pseudonymized and pseudonymized arms side by side, on the matched set I'.

**How:** Both `experiment.jsonl` files are already restricted to I' (Step 1) and each arm's similarity matrices are already saved (Step 2). So this step only loads each arm's `similarities.npz`, builds the gold neighbors (other summaries of the same work), averages the per-query retrieval metrics (P@1, Hits@10, R-Precision, MAP, NDCG), and lays the two arms side by side. Nothing is recomputed from scratch, so it runs in seconds.

In [ ]:
# Retrieval metrics for one query's ranking row. gold_neighbors = indices of other summaries of the same work.
def score_query(similarity_row, gold_neighbors):
    if not gold_neighbors: return None
    n_gold = len(gold_neighbors); gold_set = set(gold_neighbors)
    order = np.argsort(-similarity_row, kind="stable")
    relevance = np.fromiter((1 if candidate in gold_set else 0 for candidate in order), dtype=np.int64, count=len(order))
    hits = np.cumsum(relevance); precision_at_k = hits / np.arange(1, len(relevance) + 1)
    average_precision = float((precision_at_k * relevance).sum() / n_gold)
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
    dcg = float((relevance * discounts).sum()); idcg = float(discounts[:n_gold].sum())
    return dict(p_at_1=float(relevance[0]), hits_at_10=float(relevance[:10].sum() > 0),
                r_precision=float(hits[n_gold-1] / n_gold), ap=average_precision, ndcg=dcg / idcg if idcg > 0 else 0.0)

# Average the five metrics over all queries, for every matrix in this arm's similarities.npz.
def evaluate_arm(experiment_dir):
    rows = [json.loads(line) for line in (experiment_dir / "experiment.jsonl").read_text(encoding="utf-8").splitlines() if line]
    work_ids = [row["wikidata_id"] for row in rows]; n_summaries = len(work_ids)
    # gold neighbors per query (same wikidata_id, excluding self); row order matches similarities.npz
    summaries_by_work = collections.defaultdict(list)
    for index, work_id in enumerate(work_ids): summaries_by_work[work_id].append(index)
    gold_neighbors = {query: [other for other in summaries_by_work[work_ids[query]] if other != query]
                      for query in range(n_summaries)}
    similarity_matrices = np.load(experiment_dir / "similarities.npz")   # diagonals are already -inf (Step 2)
    arm_metrics = {}
    for matrix_key in similarity_matrices.files:
        similarity = similarity_matrices[matrix_key]
        per_query = [metrics for query in range(n_summaries)
                     if (metrics := score_query(similarity[query], gold_neighbors[query])) is not None]
        arm_metrics[matrix_key] = {metric_name: float(np.mean([result[metric_name] for result in per_query]))
                                   for metric_name in ("p_at_1", "hits_at_10", "r_precision", "ap", "ndcg")}
    return arm_metrics

print("scoring both arms from their saved similarities.npz ...")
results_nonanon = evaluate_arm(NONANON_DIR)
results_anon    = evaluate_arm(ANON_DIR)

# Assemble Table 2 in the exact thesis row order.
table_rows = [
    ("BoW (raw text)",                "Count vectorizer",  "bow__raw_text"),
    ("TF-IDF (raw text)",             "TF-IDF vectorizer", "tfidf__raw_text"),
    ("BoW (event triggers)",          "Count vectorizer",  "bow_events__raw_text"),
    ("TF-IDF (event triggers)",       "TF-IDF vectorizer", "tfidf_events__raw_text"),
    ("Raw text",                      "Qwen3-0.6B",        "qwen3_emb_0p6b__raw_text"),
    ("Raw text",                      "E5-Mistral",        "e5_mistral__raw_text"),
    ("Raw text",                      "StoryEmbed",        "story_emb__raw_text"),
    ("Event-only",                    "Qwen3-0.6B",        "qwen3_emb_0p6b__events_only"),
    ("Event + temporal",             "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal"),
    ("Event + causal",               "Qwen3-0.6B",        "qwen3_emb_0p6b__causal"),
    ("Event + temporal + causal",    "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal_causal_independent"),
    ("Event + joint temporo-causal", "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal_causal_joint"),
    ("Event-only",                    "E5-Mistral",        "e5_mistral__events_only"),
    ("Event + temporal",             "E5-Mistral",        "e5_mistral__temporal"),
    ("Event + causal",               "E5-Mistral",        "e5_mistral__causal"),
    ("Event + temporal + causal",    "E5-Mistral",        "e5_mistral__temporal_causal_independent"),
    ("Event + joint temporo-causal", "E5-Mistral",        "e5_mistral__temporal_causal_joint"),
]
metric_columns = [("p_at_1", "P@1"), ("hits_at_10", "Hits@10"), ("r_precision", "R-Prec."), ("ap", "MAP"), ("ndcg", "NDCG")]
table_data = []
for representation, encoder, matrix_key in table_rows:
    row = [representation, encoder]
    for arm_metrics in (results_nonanon, results_anon):
        scores = arm_metrics.get(matrix_key, {})
        row += [scores.get(metric_key, float("nan")) for metric_key, _ in metric_columns]
    table_data.append(row)
columns = pd.MultiIndex.from_tuples(
    [("", "Representation"), ("", "Vectorizer/Encoder")] +
    [("Non-Pseudonymized", metric_label) for _, metric_label in metric_columns] +
    [("Pseudonymized", metric_label) for _, metric_label in metric_columns])
matched_df = pd.DataFrame(table_data, columns=columns)

n_queries = sum(1 for line in open(NONANON_DIR / "experiment.jsonl", encoding="utf-8") if line.strip())
print(f"\nTable 2: matched retrieval over I' = {n_queries} queries (both arms), each with >=1 gold neighbor\n")
with pd.option_context("display.max_columns", None, "display.width", 240):
    print(matched_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
# matched_df holds all five metrics for both arms -> format / bold-underline for the thesis as needed.

### 4. Incremental contribution of Temporal-Causal Structural Enrichments
**Goal:** For each encoder, show how each retrieval metric changes when relation structure is added on top of the event-only representation, with the non-pseudonymized and pseudonymized arms side by side.

In [ ]:

# Check if the previous prerequisite cell is ran first
assert "results_nonanon" in globals() and "results_anon" in globals(), "Run Step 3 (Table 2) first; it computes results_nonanon / results_anon."

# The two encoders, the four added-structure conditions, and the five metrics (with their delta labels).
encoders = [("Qwen3-0.6B", "qwen3_emb_0p6b"), ("E5-Mistral", "e5_mistral")]
structures = [("Temporal",             "temporal"),
              ("Causal",               "causal"),
              ("Temporal + causal",    "temporal_causal_independent"),
              ("Joint temporo-causal", "temporal_causal_joint")]
metric_columns = [("p_at_1", "ΔP@1"), ("hits_at_10", "ΔHits@10"), ("r_precision", "ΔR-Prec."),
                  ("ap", "ΔMAP"), ("ndcg", "ΔNDCG")]

# Change in a metric from adding structure, vs the same encoder's events_only, within one arm
def structure_delta(arm_metrics, encoder, condition, metric_key):
    return (arm_metrics.get(f"{encoder}__{condition}", {}).get(metric_key, float("nan"))
            - arm_metrics.get(f"{encoder}__events_only", {}).get(metric_key, float("nan")))

# One row per (encoder, added structure): the deltas for the non-pseudonymized arm, then the pseudonymized arm
table_data = []
for encoder_label, encoder in encoders:
    for structure_label, condition in structures:
        row = [encoder_label, structure_label]
        for arm_metrics in (results_nonanon, results_anon):
            row += [structure_delta(arm_metrics, encoder, condition, metric_key) for metric_key, _ in metric_columns]
        table_data.append(row)

# Two-level header: a Non-Pseudonymized block and a Pseudonymized block, each with the five delta metrics
columns = pd.MultiIndex.from_tuples(
    [("", "Encoder"), ("", "Added structure")] +
    [("Non-Pseudonymized", metric_label) for _, metric_label in metric_columns] +
    [("Pseudonymized", metric_label) for _, metric_label in metric_columns])
ablation_df = pd.DataFrame(table_data, columns=columns)

# Print the table (positive = structure improves retrieval; negative = it degrades)
print("Incremental retrieval change from adding relation structure to event-only (matched I')")
print("(delta = condition - events_only, same encoder and arm; + = improvement)\n")
with pd.option_context("display.max_columns", None, "display.width", 320):
    print(ablation_df.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

### 5. Event-set retrieval: the Tversky diagnostic

**Goal:** Ask whether event overlap *alone* can retrieve other summaries of the same work, with no relations and no embeddings. Each summary is represented purely as a set of its extracted events.

**Two set representations:**
- **Event types:** the set of distinct MAVEN categories (abstract; e.g. {Attack, Statement, Motion}).
- **Event triggers:** the set of distinct trigger words (lexical; e.g. {attacked, killed, fled}).

**Similarity:** the symmetric Tversky index `T(A, B) = |A ∩ B| / (|A ∩ B| + α|A \ B| + β|B \ A|)` with `α = β = 0.5`, which equals the Dice coefficient.

**How:** Reuse the matched set I' from Step 1 (no re-intersection). For each arm, read the extracted events from `tma_subset_events_processed.test.jsonl`, build the two set representations, score them with the same per-query retrieval protocol used for Table 2, and place the non-pseudonymized and pseudonymized blocks side by side.



In [ ]:

# Reuse the matched set built in Step 1 (no re-intersection); sort for a deterministic row order.
assert "Iprime" in globals(), "Run Step 1 first; it builds the matched set Iprime."
matched_keys = sorted(Iprime)
ALPHA = BETA = 0.5                       # symmetric Tversky == Dice coefficient
id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')

# Per-query retrieval metrics for one ranking row (gold = other summaries of the same work).
def query_metrics(similarity_row, gold_neighbors):
    if not gold_neighbors: return None
    n_gold = len(gold_neighbors); gold_set = set(gold_neighbors)
    order = np.argsort(-similarity_row, kind="stable")
    relevance = np.fromiter((1 if candidate in gold_set else 0 for candidate in order), dtype=np.int64, count=len(order))
    hits = np.cumsum(relevance); precision_at_k = hits / np.arange(1, len(relevance) + 1)
    average_precision = float((precision_at_k * relevance).sum() / n_gold)
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
    dcg = float((relevance * discounts).sum()); idcg = float(discounts[:n_gold].sum())
    return {"P@1": float(relevance[0]), "Hits@10": float(relevance[:10].sum() > 0), "R-Prec.": float(hits[n_gold-1] / n_gold),
            "MAP": average_precision, "NDCG": dcg / idcg if idcg > 0 else 0.0}

# Score one arm: read its extracted events for the matched keys, build two set representations, rank by Tversky.
def tversky_stats(events_path, keys):
    # Events files are far smaller than experiment.jsonl, so read them directly and keep only matched keys.
    key_to_index = {key: index for index, key in enumerate(keys)}; key_set = set(keys); n_summaries = len(keys)
    work_ids = [None] * n_summaries; events_per_summary = [None] * n_summaries
    with open(events_path, encoding="utf-8") as events_file:
        for line in events_file:
            match = id_pattern.search(line[:160])
            if not match: continue
            key = (match.group(1), match.group(2))
            if key not in key_set: continue
            record = json.loads(line); index = key_to_index[key]
            work_ids[index] = key[0]; events_per_summary[index] = record.get("events", [])

    # Two set representations per summary: abstract MAVEN categories, and lexical trigger words.
    representations = {
        "Event types (MAVEN categories)": [{event["event_type"] for event in events} for events in events_per_summary],
        "Events (trigger words)":         [{event["trigger"].lower().strip() for event in events} for events in events_per_summary],
    }

    # Gold neighbors per query (same wikidata_id, excluding self).
    summaries_by_work = collections.defaultdict(list)
    for index, work_id in enumerate(work_ids): summaries_by_work[work_id].append(index)
    gold_neighbors = {query: [other for other in summaries_by_work[work_ids[query]] if other != query]
                      for query in range(n_summaries)}

    metric_names = ["P@1", "Hits@10", "R-Prec.", "MAP", "NDCG"]
    representation_stats = {}
    for representation_name, summary_sets in representations.items():
        # One-hot the set membership, then compute the symmetric Tversky similarity over all pairs (formula in the markdown above).
        vocabulary = sorted({item for summary_set in summary_sets for item in summary_set})
        vocab_index = {item: position for position, item in enumerate(vocabulary)}
        membership = np.zeros((n_summaries, len(vocabulary)), dtype=np.float32)
        for index, summary_set in enumerate(summary_sets):
            for item in summary_set: membership[index, vocab_index[item]] = 1.0
        intersection = membership @ membership.T
        set_sizes = membership.sum(1); size_a, size_b = set_sizes[:, None], set_sizes[None, :]
        similarity = (intersection / (intersection + ALPHA * (size_a - intersection) + BETA * (size_b - intersection))).astype(np.float32)
        np.fill_diagonal(similarity, -np.inf)
        # Average the metrics over queries, plus descriptive set-size and similarity stats.
        per_query = [metrics for query in range(n_summaries)
                     if (metrics := query_metrics(similarity[query], gold_neighbors[query])) is not None]
        item_counts = np.array([len(summary_set) for summary_set in summary_sets])
        off_diagonal = similarity[np.isfinite(similarity)]
        representation_stats[representation_name] = {
            "Vocab size": len(vocabulary), "Mean items": float(item_counts.mean()), "SD items": float(item_counts.std()),
            "Min": int(item_counts.min()), "Max": int(item_counts.max()),
            "Mean Sim.": float(off_diagonal.mean()), "SD Sim.": float(off_diagonal.std()),
            **{metric_name: float(np.mean([result[metric_name] for result in per_query])) for metric_name in metric_names}}
    return representation_stats

# Score both arms on the SAME matched keys.
print(f"computing the Tversky diagnostic on the matched set I' = {len(matched_keys)} summaries ...")
stats_nonanon = tversky_stats(NONANON_DIR / "tma_subset_events_processed.test.jsonl", matched_keys)
stats_anon    = tversky_stats(ANON_DIR    / "tma_subset_events_processed.test.jsonl", matched_keys)

# Assemble the table: a Non-Pseudonymized block and a Pseudonymized block, identical columns.
descriptive_columns = ["Vocab size", "Mean items", "SD items", "Min", "Max", "Mean Sim.", "SD Sim."]
metric_columns = ["P@1", "Hits@10", "R-Prec.", "MAP", "NDCG"]
formats = {"Vocab size": "{:.0f}", "Mean items": "{:.1f}", "SD items": "{:.1f}", "Min": "{:.0f}", "Max": "{:.0f}",
           "Mean Sim.": "{:.3f}", "SD Sim.": "{:.3f}", **{column: "{:.4f}" for column in metric_columns}}

# One block (two rows) per arm, formatted for display.
def build_block(condition, representation_stats):
    rows = []
    for representation_name in ["Event types (MAVEN categories)", "Events (trigger words)"]:
        values = representation_stats[representation_name]
        row = {"Condition": condition, "Story representation": representation_name}
        row.update({column: formats[column].format(values[column]) for column in descriptive_columns + metric_columns})
        rows.append(row)
    return rows

table_rows = build_block("Non-Pseudonymized", stats_nonanon) + build_block("Pseudonymized", stats_anon)
tversky_df = pd.DataFrame(table_rows)[["Condition", "Story representation"] + descriptive_columns + metric_columns]
print(f"\nSet-based Tversky index (a=b={ALPHA}, Dice) | matched I' = {len(matched_keys)} summaries / {len({work_id for work_id, _ in matched_keys})} stories\n")
with pd.option_context("display.max_columns", None, "display.width", 240):
    print(tversky_df.to_string(index=False))